# London House Price Prediction Notebook

This notebook covers four sections:

1. Data ingestion, schema checks, and join validation
2. Missing-value handling and dataset preparation
3. Exploratory data analysis with housing-relevant plots
4. Baseline modelling, evaluation, and an improved engineered model


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11


## 1. Data ingestion + schema checks

The target is `price`. We join area-level features onto each property via `outcode` so every row contains both home-level and neighbourhood-level signals.

In [ ]:
DATA_DIR = Path(".")
if not (DATA_DIR / "london_house_prices.csv").exists():
    DATA_DIR = Path("codex")

prices_path = DATA_DIR / "london_house_prices.csv"
areas_path = DATA_DIR / "london_area_features.csv"

prices = pd.read_csv(prices_path)
areas = pd.read_csv(areas_path)

print(f"prices shape: {prices.shape}")
print(f"areas shape:  {areas.shape}")
display(prices.head())
display(areas.head())


In [ ]:
required_price_cols = {
    "outcode", "latitude", "longitude", "bedrooms", "bathrooms", "floorAreaSqM",
    "livingRooms", "propertyType", "tenure", "energyRating", "rentEstimate", "price"
}
required_area_cols = {
    "outcode", "outcode_lat", "outcode_lon", "n_properties", "crime_total",
    "census_denom_total", "poi_total"
}

missing_price_cols = sorted(required_price_cols - set(prices.columns))
missing_area_cols = sorted(required_area_cols - set(areas.columns))

assert not missing_price_cols, f"Missing price columns: {missing_price_cols}"
assert not missing_area_cols, f"Missing area columns: {missing_area_cols}"

prices["outcode"] = prices["outcode"].astype(str).str.strip().str.upper()
areas["outcode"] = areas["outcode"].astype(str).str.strip().str.upper()

schema_report = pd.DataFrame({
    "prices_dtype": prices.dtypes.astype(str),
    "prices_missing": prices.isna().sum(),
}).sort_index()
display(schema_report)

area_schema_report = pd.DataFrame({
    "areas_dtype": areas.dtypes.astype(str),
    "areas_missing": areas.isna().sum(),
}).sort_index()
display(area_schema_report)


In [ ]:
range_checks = {
    "price_non_negative": (prices["price"] >= 0).all(),
    "lat_valid": prices["latitude"].dropna().between(51.2, 51.8).all(),
    "lon_valid": prices["longitude"].dropna().between(-0.6, 0.3).all(),
    "area_lat_valid": areas["outcode_lat"].dropna().between(51.2, 51.8).all(),
    "area_lon_valid": areas["outcode_lon"].dropna().between(-0.6, 0.3).all(),
}

duplicates_report = pd.Series({
    "price_full_row_duplicates": int(prices.duplicated().sum()),
    "area_outcode_duplicates": int(areas["outcode"].duplicated().sum()),
    "price_outcode_unique": int(prices["outcode"].nunique()),
    "area_outcode_unique": int(areas["outcode"].nunique()),
})

print("Range checks")
display(pd.Series(range_checks, name="passed"))
print("Duplicates / keys")
display(duplicates_report)


In [ ]:
df = prices.merge(areas, on="outcode", how="left", indicator=True, validate="many_to_one")
join_counts = df["_merge"].value_counts(dropna=False).rename_axis("join_status").to_frame("rows")
join_counts["share"] = join_counts["rows"] / len(df)
display(join_counts)

unmatched_outcodes = df.loc[df["_merge"] != "both", "outcode"].drop_duplicates().sort_values()
print(f"Unmatched outcodes: {len(unmatched_outcodes)}")
if len(unmatched_outcodes) > 0:
    display(unmatched_outcodes.head(20))

df = df.drop(columns="_merge")


## 2. Missing-value handling and preparation

The property table has visible sparsity in home-specific columns such as room counts, floor area, tenure, and EPC rating. The area table is much denser and mostly acts as enrichment keyed by `outcode`.

Handling strategy:

- Keep raw missingness visible during EDA
- Add missing-indicator flags for key property attributes
- Use median imputation for numeric fields
- Use most-frequent imputation for categorical fields
- Preserve all rows rather than dropping observations aggressively


In [ ]:
missing_summary = (
    df.isna().mean()
    .sort_values(ascending=False)
    .rename("missing_share")
    .to_frame()
)
missing_summary["missing_pct"] = (missing_summary["missing_share"] * 100).round(2)
display(missing_summary.head(25))


In [ ]:
df_model = df.copy()

for col in ["bedrooms", "bathrooms", "floorAreaSqM", "livingRooms", "energyRating", "tenure"]:
    df_model[f"{col}_missing"] = df_model[col].isna().astype(int)

df_model["price_log1p"] = np.log1p(df_model["price"])

print(df_model.shape)
display(df_model.filter(regex="_missing$").mean().sort_values(ascending=False).to_frame("share_missing_flag"))


## 3. EDA and insight generation

These plots focus on patterns that matter for a house-price use case: target skew, size-to-price relationship, premium differences across property types, and the influence of neighbourhood variables such as crime, amenities, and geography.

In [ ]:
eda_sample = df_model.sample(n=min(30000, len(df_model)), random_state=42).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_model["price"], bins=80, ax=axes[0], color="#4C78A8")
axes[0].set_title("Sale price distribution")
axes[0].set_xlabel("Price")

sns.histplot(df_model["price_log1p"], bins=80, ax=axes[1], color="#F58518")
axes[1].set_title("Log-transformed sale price distribution")
axes[1].set_xlabel("log1p(price)")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=eda_sample,
    x="floorAreaSqM",
    y="price",
    hue="propertyType",
    alpha=0.35,
    s=30,
    ax=axes[0],
    legend=False,
)
axes[0].set_title("Floor area vs price")
axes[0].set_xlabel("Floor area (sq m)")
axes[0].set_ylabel("Price")

property_order = (
    df_model.groupby("propertyType")["price"]
    .median()
    .sort_values(ascending=False)
    .head(12)
    .index
)
sns.boxplot(
    data=df_model[df_model["propertyType"].isin(property_order)],
    y="propertyType",
    x="price",
    order=property_order,
    ax=axes[1],
    color="#72B7B2",
)
axes[1].set_title("Median price spread by property type")
axes[1].set_xlabel("Price")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=eda_sample,
    x="crime_total",
    y="price",
    alpha=0.3,
    ax=axes[0],
    color="#E45756",
)
axes[0].set_title("Area crime volume vs price")
axes[0].set_xlabel("crime_total")
axes[0].set_ylabel("Price")

hb = axes[1].hexbin(
    eda_sample["longitude"],
    eda_sample["latitude"],
    C=eda_sample["price"],
    reduce_C_function=np.median,
    gridsize=35,
    cmap="viridis",
)
axes[1].set_title("Median price by location")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
plt.colorbar(hb, ax=axes[1], label="Median price")
plt.tight_layout()
plt.show()


In [ ]:
corr_cols = [
    "price", "rentEstimate", "floorAreaSqM", "bedrooms", "bathrooms", "livingRooms",
    "crime_total", "n_properties", "census_level4_perc", "census_no_qualifications_perc", "poi_total"
]
corr = df_model[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap for key numeric features")
plt.show()


In [ ]:
top_property_types = (
    df_model.groupby("propertyType")
    .agg(median_price=("price", "median"), listings=("price", "size"))
    .sort_values("median_price", ascending=False)
    .head(10)
)

top_outcodes = (
    df_model.groupby("outcode")
    .agg(median_price=("price", "median"), listings=("price", "size"))
    .query("listings >= 100")
    .sort_values("median_price", ascending=False)
    .head(10)
)

print("Top property types by median price")
display(top_property_types)
print("Top outcodes by median price (min 100 listings)")
display(top_outcodes)

print("Quick insight prompts")
print("- Prices are usually right-skewed, so log(price) is easier to model.")
print("- Floor area and rent estimate usually behave as the strongest direct price signals.")
print("- Location remains a major driver even after controlling for property size.")
print("- Area features are better interpreted as context variables than standalone causal drivers.")


## 4. Baseline model training and evaluation harness

The baseline below is a regularized linear model. It gives a simple, transparent benchmark after imputation, scaling, and one-hot encoding.

The evaluation harness reports metrics on the original price scale:

- RMSE
- MAE
- R²


In [ ]:
target = "price"
drop_cols = ["price", "price_log1p"]
X = df_model.drop(columns=drop_cols, errors="ignore")
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

baseline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler(with_mean=False)),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    fitted = clone(model)
    fitted.fit(X_train, y_train)
    preds = fitted.predict(X_test)
    return fitted, {
        "model": name,
        "rmse": mean_squared_error(y_test, preds, squared=False),
        "mae": mean_absolute_error(y_test, preds),
        "r2": r2_score(y_test, preds),
    }


In [ ]:
dummy_model = DummyRegressor(strategy="median")

baseline_model = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", baseline_preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]),
    func=np.log1p,
    inverse_func=np.expm1,
)

dummy_fitted, dummy_metrics = evaluate_model("Dummy median", dummy_model, X_train, X_test, y_train, y_test)
baseline_fitted, baseline_metrics = evaluate_model("Baseline Ridge (log target)", baseline_model, X_train, X_test, y_train, y_test)

baseline_results = pd.DataFrame([dummy_metrics, baseline_metrics]).sort_values("rmse")
display(baseline_results)


## 5. Feature engineering, tuning, and model upgrade

To improve on the linear baseline, we create features that better reflect housing economics and local context.

Engineered examples:

- `rent_to_price_proxy`: rent estimate divided by price proxy inputs after imputation-ready handling
- `bed_bath_ratio`: bedroom-to-bathroom balance
- `rooms_per_sqm`: intensity of room usage
- `crime_per_property`: crime volume adjusted by local stock count
- `poi_per_property`: amenities adjusted by local stock count
- `distance_to_central_london`: approximate distance from a central reference point

The upgraded model changes from linear regression to a tuned random forest so it can capture non-linear interactions between property attributes and neighbourhood context.

In [ ]:
def engineer_features(frame):
    data = frame.copy()
    eps = 1e-6

    data["bed_bath_ratio"] = data["bedrooms"] / (data["bathrooms"] + 1)
    data["rooms_per_sqm"] = data["livingRooms"] / (data["floorAreaSqM"] + 1)
    data["bedrooms_per_sqm"] = data["bedrooms"] / (data["floorAreaSqM"] + 1)
    data["rent_to_area_ratio"] = data["rentEstimate"] / (data["floorAreaSqM"] + 1)
    data["crime_per_property"] = data["crime_total"] / (data["n_properties"] + 1)
    data["poi_per_property"] = data["poi_total"] / (data["n_properties"] + 1)
    data["education_advantage"] = data["census_level4_perc"] - data["census_no_qualifications_perc"]

    centre_lat, centre_lon = 51.5074, -0.1278
    data["distance_to_central_london"] = np.sqrt(
        (data["latitude"] - centre_lat) ** 2 + (data["longitude"] - centre_lon) ** 2
    )

    inf_cols = data.columns[data.isin([np.inf, -np.inf]).any()].tolist()
    if inf_cols:
        data[inf_cols] = data[inf_cols].replace([np.inf, -np.inf], np.nan)

    return data

feature_engineer = FunctionTransformer(engineer_features, validate=False)
X_train_eng = engineer_features(X_train)
eng_numeric_features = X_train_eng.select_dtypes(include=[np.number]).columns.tolist()
eng_categorical_features = X_train_eng.select_dtypes(exclude=[np.number]).columns.tolist()

improved_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            eng_numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            eng_categorical_features,
        ),
    ]
)

improved_pipeline = Pipeline([
    ("feature_engineering", feature_engineer),
    ("preprocessor", improved_preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

improved_model = TransformedTargetRegressor(
    regressor=improved_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
)


In [ ]:
search_space = {
    "regressor__model__n_estimators": [200, 300, 500],
    "regressor__model__max_depth": [15, 25, None],
    "regressor__model__min_samples_split": [2, 5, 10],
    "regressor__model__min_samples_leaf": [1, 2, 4],
    "regressor__model__max_features": ["sqrt", 0.5, None],
}

tune_size = min(80000, len(X_train))
X_tune = X_train.sample(n=tune_size, random_state=42)
y_tune = y_train.loc[X_tune.index]

random_search = RandomizedSearchCV(
    estimator=improved_model,
    param_distributions=search_space,
    n_iter=10,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1,
)

random_search.fit(X_tune, y_tune)
best_improved_model = random_search.best_estimator_
print(random_search.best_params_)


In [ ]:
best_improved_model.fit(X_train, y_train)
improved_preds = best_improved_model.predict(X_test)

improved_metrics = {
    "model": "Engineered + tuned RandomForest",
    "rmse": mean_squared_error(y_test, improved_preds, squared=False),
    "mae": mean_absolute_error(y_test, improved_preds),
    "r2": r2_score(y_test, improved_preds),
}

comparison = pd.DataFrame([dummy_metrics, baseline_metrics, improved_metrics]).sort_values("rmse")
display(comparison)


In [ ]:
best_pipeline = best_improved_model.regressor_
rf_model = best_pipeline.named_steps["model"]
preprocessor = best_pipeline.named_steps["preprocessor"]

feature_names = preprocessor.get_feature_names_out()
importances = pd.Series(rf_model.feature_importances_, index=feature_names)
top_features = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x=top_features.values, y=top_features.index, color="#4C78A8")
plt.title("Top feature importances from tuned random forest")
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

display(top_features.to_frame("importance"))


## Summary

This workflow gives you a complete notebook for the use case:

- Section 1 validates both source files and the property-to-area join
- Section 2 quantifies and handles missingness without throwing away most of the data
- Section 3 uses plots aligned with housing-price analysis rather than generic EDA
- Section 4 establishes a transparent baseline and then improves it with engineered neighbourhood and property interaction features plus model tuning

If runtime is high on your machine, reduce `eda_sample`, `tune_size`, or the random-search `n_iter` value first.